In [1]:
import requests
import torch
import re
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [2]:
ds = load_dataset("dair-ai/emotion", "split")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    2000 non-null   object
 1   label   2000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 31.4+ KB


In [3]:
test['label'] = test['label'].map({0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'})

labels = test['label'].unique()

test

,text,label
0,im feeling rather rotten so im not very ambiti...,sadness
1,im updating my blog because i feel shitty,sadness
2,i never make her separate from me because i do...,sadness
3,i left with my bouquet of red and yellow tulip...,joy
4,i was feeling a little vain when i did this one,sadness
...,...,...
1995,i just keep feeling like someone is being unki...,anger
1996,im feeling a little cranky negative after this...,anger
1997,i feel that i am useful to my people and that ...,joy
1998,im feeling more comfortable with derby i feel ...,joy


In [4]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_26168\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


50831360

In [5]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [6]:
def classify(text, labels):
    url = "http://localhost:11434/api/chat"
    
    messages = [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multiclass classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of tweets. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
        ]
    
    start_time = time.time()

    try:
        response = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": True,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 3100
            }
        }, timeout=30)
        response_time = time.time() - start_time
        vram_usage = get_gpu_memory_usage()
        ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)
        response = response.json()
        response_text = response['message'].get('thinking', '') if 'message' in response else ''
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return "error", {}, 0, 0, 0, 0, f"API Error: {e}"
    
    if not response.get('done', False):
        print(f"Ollama returned an incomplete response: {response.get('error')}")
        return 'error', {}, response_time, 0, 0, 0, response.get('error', 'Incomplete response')
    
    if 'message' in response and 'content' in response['message']:
        classification_text = response['message']['content'].lower()
        print("Response fields:", ', '.join(response.keys()))
        print(response)
        total_time = response['total_duration'] / 1_000_000_000
    else:
        messages.append({"role": "assistant", "content": response_text + '</think>'})
        start_time2 = time.time()
        response2 = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": False,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 430
            }
        })
        response_time += time.time() - start_time2
        response2 = response2.json()
        classification_text = response2['message']['content'].lower() if 'message' in response2 and 'content' in response2['message'] else ''
        total_time = response['total_duration'] / 1_000_000_000 + response2['total_duration'] / 1_000_000_000
    
    label_counts = {label: len(re.findall(r'\b' + re.escape(label.lower()) + r'\b', classification_text)) for label in labels}
    
    if all(count == label_counts[labels[0]] for count in label_counts.values()):
        content = 'error'
    else:
        content = max(label_counts, key=label_counts.get)
    
    print(f"Text: {text}")
    print(f"Response: {content}")
    
    return content, label_counts, response_time, vram_usage, ram_usage_bytes, total_time, response_text

In [7]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'label_counts','response_time', 'vram_usage', 'ram_usage', 'total_time', 'response_text']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_26168\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


Response fields: model, created_at, message, done_reason, done, total_duration, load_duration, prompt_eval_count, prompt_eval_duration, eval_count, eval_duration
{'model': 'deepseek-r1:1.5b', 'created_at': '2025-06-21T18:33:17.3607238Z', 'message': {'role': 'assistant', 'content': 'sadness', 'thinking': 'Okay, I need to classify the given text into one of the specified labels related to sentiment analysis of tweets. The task is to determine whether the text falls under sadness, joy, fear, anger, love, or surprise.\n\nThe text provided is: "im feeling rather rotten so im not very ambitious right now"\n\nFirst, let\'s break down the sentence. It starts with "im feeling," which suggests a negative emotion. "Rotten" implies that something is bad or unpleasant. The phrase "not very ambitious" indicates that the person isn\'t confident in their abilities.\n\nNow, considering the labels:\n\n- Sadness: This would typically involve intense emotions like anger or fear.\n- Joy: Positive feelings,

In [8]:
test.to_csv('results/deepseekR1_ZS_multiclass3.csv', index=False)
test

,text,label,prediction,label_counts,response_time,vram_usage,ram_usage,total_time,response_text
0,im feeling rather rotten so im not very ambiti...,sadness,sadness,"{'sadness': 1, 'joy': 0, 'fear': 0, 'anger': 0...",7.947927,2376,89.593750,5.875109,"Okay, I need to classify the given text into o..."
1,im updating my blog because i feel shitty,sadness,sadness,"{'sadness': 3, 'joy': 0, 'fear': 0, 'anger': 0...",5.907937,2367,89.859375,3.862112,"Okay, so I need to figure out how to classify ..."
2,i never make her separate from me because i do...,sadness,love,"{'sadness': 0, 'joy': 0, 'fear': 0, 'anger': 0...",6.201511,2376,89.820312,4.180675,"Okay, so I need to figure out how to classify ..."
3,i left with my bouquet of red and yellow tulip...,joy,joy,"{'sadness': 1, 'joy': 2, 'fear': 0, 'anger': 0...",7.229880,2356,90.117188,5.195867,"Okay, so I need to classify this tweet into on..."
4,i was feeling a little vain when i did this one,sadness,love,"{'sadness': 0, 'joy': 0, 'fear': 0, 'anger': 0...",6.228651,2372,90.746094,4.189808,"Okay, so I need to figure out how to classify ..."
...,...,...,...,...,...,...,...,...,...
1995,i just keep feeling like someone is being unki...,anger,anger,"{'sadness': 0, 'joy': 0, 'fear': 0, 'anger': 2...",4.244250,2308,76.582031,2.201989,"Okay, so I need to figure out how to classify ..."
1996,im feeling a little cranky negative after this...,anger,anger,"{'sadness': 1, 'joy': 1, 'fear': 1, 'anger': 3...",6.763696,2304,77.054688,4.716883,"Okay, so I need to figure out how to classify ..."
1997,i feel that i am useful to my people and that ...,joy,joy,"{'sadness': 0, 'joy': 2, 'fear': 0, 'anger': 0...",4.131793,2308,77.066406,2.092654,"Okay, so I need to figure out how to classify ..."
1998,im feeling more comfortable with derby i feel ...,joy,joy,"{'sadness': 0, 'joy': 2, 'fear': 0, 'anger': 0...",4.409795,2308,76.750000,2.357337,"Okay, so I need to figure out how to classify ..."


In [9]:
y_pred = test['prediction']
y_true = test['label']

#import acc, f1_score, precision and recall from sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.341500
F1 score: 0.378788
Precision: 0.552938
Recall: 0.341500


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 5.379193484306335
Average VRAM usage: 2333.929
Average RAM usage: 77.5478359375
Average total time: 3.33566496775


In [11]:
# save results to txt
with open('results/deepseekR1_ZS_multiclass3.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')
    f.write(f'Lines classified: {len(test)}\n')